In [9]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle
from sklearn.tree import DecisionTreeClassifier

from sklearn.linear_model import LogisticRegression

print("✅ Libraries loaded successfully.")

✅ Libraries loaded successfully.


In [5]:
# ==============================
# STEP 2: LOAD SAFE AND PHISHING DATASETS (FINAL VERSION)
# ==============================

import pandas as pd

# --- Load Safe URLs from Excel ---
safe_domains = pd.read_excel("dataset/safe_sites.xlsx")

# Normalize column names just in case (e.g., "Domain", "domain", etc.)
safe_domains.columns = [c.strip().lower() for c in safe_domains.columns]

if 'domain' in safe_domains.columns:
    safe_domains['url'] = "https://www." + safe_domains['domain'].astype(str)
elif 'url' in safe_domains.columns:
    safe_domains['url'] = safe_domains['url'].astype(str)
else:
    raise ValueError("❌ No valid 'Domain' or 'URL' column found in safe_sites.xlsx")

safe_domains['label'] = 0  # Safe label
safe = safe_domains[['url', 'label']].drop_duplicates(subset=['url']).dropna()

print("✅ Safe URLs loaded:", len(safe))
print(safe.head(), "\n")


# --- Load PhishTank URLs ---
phish = pd.read_csv("dataset/phishtank.csv")

# Try to detect correct URL column automatically
if 'url' not in phish.columns:
    possible_cols = [c for c in phish.columns if 'url' in c.lower()]
    if possible_cols:
        phish = phish.rename(columns={possible_cols[0]: 'url'})
    else:
        raise ValueError("❌ No column containing 'url' found in phishtank.csv!")

phish = phish[['url']].dropna()
phish['label'] = 1  # Phishing label
phish = phish.drop_duplicates(subset=['url'])

print("✅ Phishing URLs loaded:", len(phish))
print(phish.head(), "\n")


# --- Combine Safe + Phishing Datasets ---
df = pd.concat([safe, phish], ignore_index=True).drop_duplicates(subset=['url'])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle rows

print("✅ Combined Dataset Shape:", df.shape)
print(df['label'].value_counts())
df.head()


✅ Safe URLs loaded: 1048575
                                url  label
0          https://www.facebook.com      0
1  https://www.fonts.googleapis.com      0
2         https://www.instagram.com      0
3  https://www.googletagmanager.com      0
4            https://www.google.com      0 

✅ Phishing URLs loaded: 49068
                                                 url  label
0                https://1eetwsep.forms.app/midinero      1
1  https://joelosteen12645-creator.github.io/ok/#...      1
2                https://safesupport-reprog-be.info/      1
3            https://www.safesupport-reprog-be.info/      1
4                     http://v-vpass-jp.tjweidu.com/      1 

✅ Combined Dataset Shape: (1097643, 2)
label
0    1048575
1      49068
Name: count, dtype: int64


,url,label
0,https://www.nywf64.com,0
1,https://www.gtomega.eu,0
2,https://www.colsearch.nfsa.afc.gov.au,0
3,https://docs.google.com/presentation/d/e/2PACX...,1
4,https://www.njpw.com,0


In [6]:
# ================== STEP 3: FEATURE EXTRACTION ==================

def extract_features(url):
    parsed = urlparse(str(url))
    host = parsed.netloc.lower() or ""
    lower_url = str(url).lower()

    features = {
        'url_length': len(lower_url),
        'domain_length': len(host),
        'num_dots': lower_url.count('.'),
        'num_hyphens': lower_url.count('-'),
        'num_slashes': lower_url.count('/'),
        'num_digits': sum(c.isdigit() for c in lower_url),
        'has_https': 1 if parsed.scheme == 'https' else 0,
        'has_at_symbol': 1 if '@' in lower_url else 0,
        'has_ip_address': 1 if re.search(r'(\d{1,3}\.){3}\d{1,3}', host) else 0,
        'num_subdomains': len(host.split('.')) - 2 if len(host.split('.')) > 2 else 0,
        'has_suspicious_word': 1 if re.search(
            r'login|secure|account|update|verify|bank|free|confirm|signin',
            lower_url) else 0,
        'has_long_domain': 1 if len(host) > 30 else 0
    }
    return features


# Apply to all URLs
print("⏳ Extracting features...")
feature_data = [extract_features(url) for url in df["url"]]
X = pd.DataFrame(feature_data)
y = df["label"]

print("✅ Features extracted:", X.shape)
X.head()


⏳ Extracting features...
✅ Features extracted: (1097643, 12)


,url_length,domain_length,num_dots,num_hyphens,num_slashes,num_digits,has_https,has_at_symbol,has_ip_address,num_subdomains,has_suspicious_word,has_long_domain
0,22,14,2,0,2,2,1,0,0,1,0,0
1,22,14,2,0,2,0,1,0,0,1,0,0
2,37,29,5,0,2,0,1,0,0,4,0,0
3,131,15,2,4,7,14,1,0,0,1,0,0
4,20,12,2,0,2,0,1,0,0,1,0,0


In [7]:
# ================== STEP 4: TRAIN TEST SPLIT ==================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print("✅ Data split into train/test sets.")
print("Training samples:", len(X_train), "| Testing samples:", len(X_test))


✅ Data split into train/test sets.
Training samples: 768350 | Testing samples: 329293


In [10]:
# ================== STEP 4.1: COMPARE MULTIPLE MODELS ==================
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Initialize all three models
model_dt = DecisionTreeClassifier(random_state=42)
model_rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, class_weight='balanced')
model_lr = LogisticRegression(max_iter=500, solver='liblinear', random_state=42)

# Train them
model_dt.fit(X_train, y_train)
model_rf.fit(X_train, y_train)
model_lr.fit(X_train, y_train)

# Predict on test data
pred_dt = model_dt.predict(X_test)
pred_rf = model_rf.predict(X_test)
pred_lr = model_lr.predict(X_test)

# Accuracy comparison
acc_dt = accuracy_score(y_test, pred_dt)
acc_rf = accuracy_score(y_test, pred_rf)
acc_lr = accuracy_score(y_test, pred_lr)

print("===============================================")
print(f"📘 Decision Tree Accuracy:      {acc_dt:.4f}")
print(f"🌲 Random Forest Accuracy:       {acc_rf:.4f}")
print(f"📈 Logistic Regression Accuracy: {acc_lr:.4f}")
print("===============================================")

# Optional detailed reports
print("\n--- Decision Tree ---")
print(classification_report(y_test, pred_dt))

print("\n--- Random Forest ---")
print(classification_report(y_test, pred_rf))

print("\n--- Logistic Regression ---")
print(classification_report(y_test, pred_lr))


📘 Decision Tree Accuracy:      0.9994
🌲 Random Forest Accuracy:       0.9990
📈 Logistic Regression Accuracy: 0.9991

--- Decision Tree ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    314573
           1       1.00      0.99      0.99     14720

    accuracy                           1.00    329293
   macro avg       1.00      0.99      1.00    329293
weighted avg       1.00      1.00      1.00    329293


--- Random Forest ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    314573
           1       0.99      0.99      0.99     14720

    accuracy                           1.00    329293
   macro avg       0.99      0.99      0.99    329293
weighted avg       1.00      1.00      1.00    329293


--- Logistic Regression ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    314573
           1       1.00      0.98      0.99   

In [11]:
# ================== STEP 5: MODEL TRAINING ==================
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    class_weight='balanced'   # helps with label imbalance
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"✅ Model trained successfully. Accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred))


✅ Model trained successfully. Accuracy: 0.9990
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    314573
           1       0.99      0.99      0.99     14720

    accuracy                           1.00    329293
   macro avg       0.99      0.99      0.99    329293
weighted avg       1.00      1.00      1.00    329293



In [12]:
# ================== STEP 6: SAVE MODEL ==================
pickle.dump(model, open("phishing_model.pkl", "wb"))
print("✅ Model saved successfully as phishing_model.pkl")


✅ Model saved successfully as phishing_model.pkl


In [13]:
# ================== STEP 7: TEST PREDICTIONS (WITH BRAND MISMATCH) ==================
import requests, socket, tldextract
from urllib.parse import urlparse

def url_exists_loose(url, timeout=5):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        r = requests.head(url, headers=headers, timeout=timeout, allow_redirects=True)
        if r.status_code < 400 or r.status_code in (403, 405):
            return True
    except requests.RequestException:
        pass
    try:
        r = requests.get(url, headers=headers, timeout=timeout, allow_redirects=True)
        if r.status_code < 400 or r.status_code in (403, 405):
            return True
    except requests.RequestException:
        pass
    try:
        host = (urlparse(url).netloc.split(":")[0]) or urlparse(url).path
        if host:
            socket.gethostbyname(host)
            # DNS-ok but HTTP blocked => still treat as unreachable for our purposes
            return False
    except Exception:
        pass
    return False

WHITELIST_REG = {
    "google.com", "instagram.com", "facebook.com", "microsoft.com",
    "wikipedia.org", "amazon.in", "github.com", "twitter.com", "bing.com",
}

# brand keyword -> official registrable domain
OFFICIAL_BRANDS = {
    "paypal": "paypal.com",
    "google": "google.com",
    "facebook": "facebook.com",
    "instagram": "instagram.com",
    "microsoft": "microsoft.com",
    "bankofamerica": "bankofamerica.com",
}

PHISH_WORDS = ["login", "verify", "secure", "account", "update", "bank", "signin"]

def registrable(host: str) -> str:
    ext = tldextract.extract(host)
    return f"{ext.domain}.{ext.suffix}" if ext.domain and ext.suffix else host.lower()

def is_whitelisted(host: str) -> bool:
    return registrable(host) in WHITELIST_REG

def has_brand_mismatch(url: str, host: str) -> bool:
    reg = registrable(host)
    low = url.lower()
    for kw, official in OFFICIAL_BRANDS.items():
        if kw in low:
            # it references a brand but is NOT that brand's registrable domain
            if reg != official and not reg.endswith("." + official):
                return True
    return False

def realistic_predict_url(url):
    try:
        if not url.startswith(("http://", "https://")):
            url = "http://" + url

        parsed = urlparse(url)
        host = parsed.netloc.lower()
        reg = registrable(host)
        reachable = url_exists_loose(url)

        # 1) Whitelist
        if reg in WHITELIST_REG:
            print(f"{url} → ✅ Safe (whitelisted)")
            return "Safe"

        # 2) Brand mismatch (strong phishing signal)
        if has_brand_mismatch(url, host):
            print(f"{url} → ⚠️ Phishing (brand mismatch)")
            return "Phishing"

        # 3) Looks-phishy keyword count
        kw_count = sum(w in url.lower() for w in PHISH_WORDS)

        # 4) If unreachable AND not phishy-looking → Non-existing
        if not reachable and kw_count == 0:
            print(f"{url} → ❌ Non-existing")
            return "Non-existing"

        # 5) Model prediction
        f = pd.DataFrame([extract_features(url)])
        expected = getattr(model, "feature_names_in_", None)
        if expected is not None:
            for col in expected:
                if col not in f:
                    f[col] = 0
            f = f[list(expected)]

        proba = float(model.predict_proba(f)[0][1])

        # Threshold with a small boost for obviously phishy patterns
        phishing = (proba > 0.7) or (kw_count >= 2 and proba > 0.3)

        label = "⚠️ Phishing" if phishing else "✅ Safe"
        print(f"{url} → {label} (prob={proba:.2f}, reachable={reachable})")
        return "Phishing" if phishing else "Safe"

    except Exception as e:
        print(f"Error checking {url}: {e}")
        return "Error"

# ---------------------- TEST SAMPLE URLs ----------------------
test_urls = [
    # ✅ SAFE
    "https://www.google.com",
    "https://www.wikipedia.org",
    "https://www.github.com",
    "https://www.microsoft.com",
    "https://www.amazon.in",

    # ⚠️ PHISHING (should now be Phishing)
    "https://secure-login-paypal-example.com/verify",
    "https://account-update-bankofamerica-login.com/login",
    "https://secure-paypal-login.google-verify.com",     # <-- brand-mismatch against google.com
    "http://login-facebook.secure-account-verify.com",
    "https://verify-your-bank-account-now.example-login.com",

    # ❌ NON-EXISTING
    "http://this-domain-definitely-does-not-exist-12345.com",
    "http://nonexistent-subdomain.example-nxdomain-test.xyz",
    "http://random-fake-site-abcdefg-9999.org",
    "http://192.0.2.123/login",
    "http://10.255.255.1/login",
]

print("\n==================== TEST RESULTS ====================")
for url in test_urls:
    realistic_predict_url(url)



==================== TEST RESULTS ====================
https://www.google.com → ✅ Safe (whitelisted)
https://www.wikipedia.org → ✅ Safe (whitelisted)
https://www.github.com → ✅ Safe (whitelisted)
https://www.microsoft.com → ✅ Safe (whitelisted)
https://www.amazon.in → ✅ Safe (whitelisted)
https://secure-login-paypal-example.com/verify → ⚠️ Phishing (brand mismatch)
https://account-update-bankofamerica-login.com/login → ⚠️ Phishing (brand mismatch)
https://secure-paypal-login.google-verify.com → ⚠️ Phishing (brand mismatch)
http://login-facebook.secure-account-verify.com → ⚠️ Phishing (brand mismatch)
https://verify-your-bank-account-now.example-login.com → ✅ Safe (prob=0.14, reachable=False)
http://this-domain-definitely-does-not-exist-12345.com → ❌ Non-existing
http://nonexistent-subdomain.example-nxdomain-test.xyz → ❌ Non-existing
http://random-fake-site-abcdefg-9999.org → ❌ Non-existing
http://192.0.2.123/login → ⚠️ Phishing (prob=0.99, reachable=False)
http://10.255.255.1/login → 